# Financial Risk Management using Explainable AI

This notebook performs exploratory data analysis (EDA) on the Freddie Mac
Single-Family Loan-Level Dataset.

Objectives:

1. Understand borrower characteristics at origination.
2. Understand post-origination borrower behaviour.
3. Identify indicators of financial stress.
4. Design a Borrower Serviceability Stress Index (BSSI) for subsequent modelling.

## Load Origination and Performance Datasets

The origination dataset contains borrower and loan characteristics at the
time of loan approval.

The performance dataset contains monthly repayment behaviour and loan status
throughout the life of the loan.

## Analysis of Origination Features

The following variables are considered important for assessing
borrower affordability, creditworthiness and collateral quality.

In [1]:
import sys
sys.path.append("..")

from src.data_loader import (
    load_origination,
    load_performance
)

# Origination Dataset Analysis

In [2]:
orig_df = load_origination(
    "../data/raw/historical_data_2018Q1.txt"
)

orig_df.shape

(296816, 32)

In [3]:
orig_df[[
    "credit_score",
    "dti",
    "ltv",
    "cltv",
    "original_upb",
    "interest_rate"
]].describe()

,credit_score,dti,ltv,cltv,original_upb,interest_rate
count,296816.000000,296816.000000,296816.000000,296816.000000,2.968160e+05,296816.000000
mean,746.489303,52.757567,74.640178,74.971383,2.343458e+05,4.387300
std,101.025120,127.421027,18.397557,18.615435,1.239277e+05,0.450953
min,427.000000,1.000000,5.000000,5.000000,7.000000e+03,0.000000
25%,713.000000,29.000000,67.000000,67.000000,1.400000e+05,4.125000
50%,753.000000,37.000000,80.000000,80.000000,2.100000e+05,4.375000
75%,784.000000,44.000000,86.000000,88.000000,3.080000e+05,4.625000
max,9999.000000,999.000000,999.000000,999.000000,1.230000e+06,6.625000


### Key Findings

1. Median credit score is approximately 753, indicating a predominantly
   prime borrower population.

2. Median DTI is approximately 37%, suggesting moderate debt burden.

3. Median LTV is approximately 80%, which is typical for residential
   mortgage lending.

4. Interest rates range between 0% and 6.625%, with a median of 4.375%.

# First Observation: The Data Needs Cleaning


| Column       | Max Value |
| ------------ | --------: |
| credit_score |      9999 |
| dti          |       999 |
| ltv          |       999 |
| cltv         |       999 |


999, 9999 means Missing,Unknown,Not Applicable


In [4]:
orig_df[
    [
        "credit_score",
        "dti",
        "ltv",
        "cltv"
    ]
].isin([999, 9999]).sum()

credit_score      28
dti             5258
ltv               14
cltv              17
dtype: int64

| Question                | Why                           |
| ----------------------- | ----------------------------- |
| How many missing FICOs? | Can we trust credit score?    |
| How many missing DTIs?  | Important for serviceability  |
| How many missing LTVs?  | Important for collateral risk |
| How many missing CLTVs? | Important for leverage        |


# The key borrower and loan attributes exhibited minimal missingness (<2%), allowing them to be retained for subsequent modeling after appropriate imputation.

In [5]:
orig_df[
    [
        "num_borrowers",
        "occupancy_status",
        "loan_purpose",
        "property_type",
        "property_state"
    ]
].head()

,num_borrowers,occupancy_status,loan_purpose,property_type,property_state
0,1,I,P,CO,NC
1,1,P,C,MH,OH
2,2,P,P,SF,OH
3,2,P,P,SF,NY
4,2,S,P,SF,MN


In [6]:
orig_df["occupancy_status"].value_counts()

occupancy_status
P    257635
I     27668
S     11513
Name: count, dtype: int64

### Findings

Most loans correspond to primary residences, indicating relatively
lower expected risk compared to investment properties.

| Code | Meaning             |
| ---- | ------------------- |
| P    | Primary Residence   |
| I    | Investment Property |
| S    | Second Home         |


In [7]:
orig_df["loan_purpose"].value_counts()

loan_purpose
P    170316
C     73439
N     53061
Name: count, dtype: int64

### Findings

The dataset contains purchase loans, cash-out refinances and
rate/term refinances, each representing different borrower behaviours.

| Code | Meaning               |
| ---- | --------------------- |
| P    | Purchase              |
| C    | Cash-Out Refinance- Borrower takes equity out of the property.   |
| N    | No Cash-Out Refinance  |


In [8]:
orig_df["property_type"].value_counts()

property_type
SF    186637
PU     83898
CO     24477
MH      1281
CP       523
Name: count, dtype: int64

### Findings

Single-family properties dominate the portfolio, followed by planned
unit developments and condominiums.

| Code | Meaning                  |
| ---- | ------------------------ |
| SF   | Single Family            |
| PU   | Planned Unit Development |
| CO   | Condominium              |
| MH   | Manufactured Housing     |
| CP   | Cooperative              |


Understanding Co-relations.

In [9]:
orig_df[
    [
        "credit_score",
        "dti",
        "ltv",
        "interest_rate",
        "original_upb"
    ]
].corr(numeric_only=True)

,credit_score,dti,ltv,interest_rate,original_upb
credit_score,1.000000,-0.050384,-0.019651,-0.115045,0.021495
dti,-0.050384,1.000000,-0.069225,0.017788,-0.083306
ltv,-0.019651,-0.069225,1.000000,0.159433,0.131419
interest_rate,-0.115045,0.017788,0.159433,1.000000,-0.052896
original_upb,0.021495,-0.083306,0.131419,-0.052896,1.000000


## Feature Classification

Based on domain knowledge, variables were classified into three categories:

### Serviceability Factors
- DTI
- Interest Rate
- Original UPB
- Number Borrowers

### Creditworthiness Factors
- Credit Score
- Occupancy Status
- Loan Purpose

### Collateral Factors
- LTV
- CLTV
- Property Type

This classification forms the conceptual foundation for the proposed
Serviceability Assessment Framework.

## Loan Performance Analysis

The performance dataset records monthly loan behaviour after origination.

In [10]:
perf_sample = load_performance(
    "../data/raw/historical_data_time_2018Q1.txt"
)

In [11]:
perf_sample[
    [
        "current_loan_delinquency_status",
        "modification_flag",
        "zero_balance_code",
        "borrower_assistance_status"
    ]
].head()

,current_loan_delinquency_status,modification_flag,zero_balance_code,borrower_assistance_status
0,0,NaN,NaN,NaN
1,0,NaN,NaN,NaN
2,0,NaN,NaN,NaN
3,0,NaN,NaN,NaN
4,0,NaN,NaN,NaN


In [12]:
perf_sample["modification_flag"].value_counts(dropna=False)

modification_flag
NaN    13773228
P         96657
Y          3199
Name: count, dtype: int64

In [13]:
perf_sample["zero_balance_code"].value_counts(dropna=False)

zero_balance_code
NaN     13646109
1.0       225613
96.0         642
16.0         267
2.0          179
9.0          144
15.0          96
3.0           34
Name: count, dtype: int64

In [14]:
perf_sample["borrower_assistance_status"].value_counts(dropna=False)

borrower_assistance_status
NaN    13687653
F        165606
T         15753
R          4072
Name: count, dtype: int64

In [16]:
for i, col in enumerate(perf_sample.iloc[0]):
    print(i, col)

0 F18Q10000001
1 201802
2 63000.0
3 0
4 0
5 360
6 nan
7 nan
8 nan
9 nan
10 5.25
11 0.0
12 nan
13 nan
14 nan
15 nan
16 nan
17 nan
18 nan
19 nan
20 nan
21 nan
22 nan
23 nan
24 nan
25 77
26 nan
27 nan
28 nan
29 nan
30 nan
31 63000.0


In [17]:
perf_sample.head(5).T

,0,1,2,3,4
loan_identifier,F18Q10000001,F18Q10000001,F18Q10000001,F18Q10000001,F18Q10000001
reporting_period,201802,201803,201804,201805,201806
current_actual_upb,63000.0,62000.0,62000.0,62000.0,62000.0
current_loan_delinquency_status,0,0,0,0,0
loan_age,0,1,2,3,4
remaining_months_to_maturity,360,359,358,357,356
repurchase_flag,NaN,NaN,NaN,NaN,NaN
modification_flag,NaN,NaN,NaN,NaN,NaN
zero_balance_code,NaN,NaN,NaN,NaN,NaN
zero_balance_effective_date,NaN,NaN,NaN,NaN,NaN


In [18]:
perf_sample.columns.tolist()

['loan_identifier',
 'reporting_period',
 'current_actual_upb',
 'current_loan_delinquency_status',
 'loan_age',
 'remaining_months_to_maturity',
 'repurchase_flag',
 'modification_flag',
 'zero_balance_code',
 'zero_balance_effective_date',
 'current_interest_rate',
 'current_deferred_upb',
 'due_date_last_paid_installment',
 'mi_recoveries',
 'net_sales_proceeds',
 'non_mi_recoveries',
 'expenses',
 'legal_costs',
 'maintenance_costs',
 'taxes_and_insurance',
 'misc_expenses',
 'actual_loss_calculation',
 'modification_cost',
 'step_modification_flag',
 'deferred_payment_plan',
 'estimated_ltv',
 'zero_balance_removal_upb',
 'delinquent_accrued_interest',
 'delinquency_due_to_disaster',
 'borrower_assistance_status',
 'current_month_modification_cost',
 'interest_bearing_upb']

In [19]:
from src.schema import PERFORMANCE_COLUMNS

In [21]:
perf_df = perf_sample

In [22]:
perf_df.columns = PERFORMANCE_COLUMNS

In [23]:
perf_df["loan_identifier"].nunique()

296816

In [24]:
loan_perf = perf_df.groupby("loan_identifier").agg(
    max_delinquency=(
        "current_loan_delinquency_status",
        "max"
    )
).reset_index()

In [25]:
loan_perf.head()

,loan_identifier,max_delinquency
0,F18Q10000001,0
1,F18Q10000002,1
2,F18Q10000003,0
3,F18Q10000004,0
4,F18Q10000005,0


In [26]:
loan_perf["max_delinquency"].value_counts().head(20)

max_delinquency
0     255143
1      21090
9       6954
2       4386
3       2779
5       1699
4       1530
6       1418
7        902
8        750
RA       165
Name: count, dtype: int64

# First Observation

Most loans are healthy.

255,143 / 296,816
≈ 86%

In [27]:
sorted(
    perf_df["current_loan_delinquency_status"]
    .astype(str)
    .unique()
)

['0',
 '1',
 '10',
 '11',
 '12',
 '13',
 '14',
 '15',
 '16',
 '17',
 '18',
 '19',
 '2',
 '20',
 '21',
 '22',
 '23',
 '24',
 '25',
 '26',
 '27',
 '28',
 '29',
 '3',
 '30',
 '31',
 '32',
 '33',
 '34',
 '35',
 '36',
 '37',
 '38',
 '39',
 '4',
 '40',
 '41',
 '42',
 '43',
 '44',
 '45',
 '46',
 '47',
 '48',
 '49',
 '5',
 '50',
 '51',
 '52',
 '53',
 '54',
 '55',
 '56',
 '57',
 '58',
 '59',
 '6',
 '60',
 '61',
 '62',
 '63',
 '64',
 '65',
 '66',
 '67',
 '68',
 '69',
 '7',
 '70',
 '71',
 '72',
 '8',
 '9',
 'RA']

In [28]:
loan_perf["max_delinquency"].value_counts().sort_index().head(20)

max_delinquency
0     255143
1      21090
2       4386
3       2779
4       1530
5       1699
6       1418
7        902
8        750
9       6954
RA       165
Name: count, dtype: int64

# The delinquency status is not a categorical label.

0 = Current
1 = 30 days delinquent
2 = 60 days delinquent
3 = 90 days delinquent
...
72 = 2160 days delinquent (~6 years)
RA = REO Acquisition / Real Estate Owned


### Delinquency Findings

1. Approximately 86% of loans never experienced delinquency.

2. Around 8.5% of loans experienced temporary delinquency
   (1–2 months).

3. A smaller subset of loans exhibited prolonged delinquency,
   indicating significant financial stress.

4. Severe delinquency (6+ months) was observed in several thousand loans.

In [29]:
loan_perf

,loan_identifier,max_delinquency
0,F18Q10000001,0
1,F18Q10000002,1
2,F18Q10000003,0
3,F18Q10000004,0
4,F18Q10000005,0
...,...,...
296811,F18Q10297464,0
296812,F18Q10297465,0
296813,F18Q10297466,0
296814,F18Q10297467,0


In [30]:
loan_perf = perf_df.groupby("loan_identifier").agg(
    max_delinquency=(
        "current_loan_delinquency_status",
        lambda x: max(
            [int(v) for v in x.astype(str) if v != "RA"]
        )
    ),
    ever_ra=(
        "current_loan_delinquency_status",
        lambda x: (x.astype(str) == "RA").any()
    ),
    ever_modified=(
        "modification_flag",
        lambda x: x.notna().any()
    ),
    ever_assistance=(
        "borrower_assistance_status",
        lambda x: x.notna().any()
    )
).reset_index()

In [31]:
loan_perf.head()

,loan_identifier,max_delinquency,ever_ra,ever_modified,ever_assistance
0,F18Q10000001,0,False,False,False
1,F18Q10000002,1,False,False,False
2,F18Q10000003,0,False,False,False
3,F18Q10000004,0,False,False,False
4,F18Q10000005,0,False,False,False


In [32]:
loan_perf[
    [
        "ever_ra",
        "ever_modified",
        "ever_assistance"
    ]
].sum()

ever_ra              165
ever_modified       3000
ever_assistance    21071
dtype: int64

### Stress Event Findings

- 3,000 loans experienced modification events.
- 21,071 loans required borrower assistance.
- 165 loans reached REO status.

Borrower assistance events were significantly more common than
loan modifications and REO outcomes.

This suggests that intervention programs may serve as an early indicator
of borrower financial stress.

## Borrower Serviceability Stress Index (BSSI)

To capture varying degrees of borrower distress, a multi-level
Borrower Serviceability Stress Index (BSSI) is proposed.

### BSSI Level 0
Healthy borrowers with no delinquency, assistance or modification events.

### BSSI Level 1
Borrowers exhibiting temporary delinquency (1–5 months).

### BSSI Level 2
Borrowers experiencing severe delinquency (6+ months) or requiring
loan modification.

### BSSI Level 3
Borrowers reaching REO status.

The BSSI serves as the primary target variable for subsequent machine
learning models.